<a href="https://colab.research.google.com/github/Sabharishraja/AD23633_Generative-AI/blob/main/HMM_Exp_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **WORD PREDICTION USING USER DATA**

In [ ]:
import nltk
import random
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tag import pos_tag, hmm

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
text = """
I love learning new things.
Machine learning is very interesting.
Data science helps students learn.
I enjoy working with data models.
Artificial intelligence is powerful.
"""

In [ ]:

sentences = sent_tokenize(text)
tokenize_words = []
for sent in sentences:
  words = word_tokenize(sent)
  tokenize_words.append(words)

tagged_sentences =[]
for sent in tokenize_words:
  tag = pos_tag(sent)
  tagged_sentences.append(tag)

for s in tagged_sentences:
  print(s)

[('I', 'PRP'), ('love', 'VBP'), ('learning', 'VBG'), ('new', 'JJ'), ('things', 'NNS'), ('.', '.')]
[('Machine', 'NN'), ('learning', 'NN'), ('is', 'VBZ'), ('very', 'RB'), ('interesting', 'JJ'), ('.', '.')]
[('Data', 'NNP'), ('science', 'NN'), ('helps', 'VBZ'), ('students', 'NNS'), ('learn', 'NN'), ('.', '.')]
[('I', 'PRP'), ('enjoy', 'VBP'), ('working', 'VBG'), ('with', 'IN'), ('data', 'NNS'), ('models', 'NNS'), ('.', '.')]
[('Artificial', 'JJ'), ('intelligence', 'NN'), ('is', 'VBZ'), ('powerful', 'JJ'), ('.', '.')]


In [ ]:
trainer = hmm.HiddenMarkovModelTrainer()
hmm_model = trainer.train_supervised(tagged_sentences)

#parts of speech tagging using hmm
predicted_tags = hmm_model.tag(tokenize_words[0])
print(predicted_tags)

for i in tokenize_words:
  predicted_tags = hmm_model.tag(i)
  print(predicted_tags)

[('I', 'PRP'), ('love', 'VBP'), ('learning', 'VBG'), ('new', 'JJ'), ('things', 'NNS'), ('.', '.')]
[('I', 'PRP'), ('love', 'VBP'), ('learning', 'VBG'), ('new', 'JJ'), ('things', 'NNS'), ('.', '.')]
[('Machine', 'NN'), ('learning', 'NN'), ('is', 'VBZ'), ('very', 'RB'), ('interesting', 'JJ'), ('.', '.')]
[('Data', 'NNP'), ('science', 'NN'), ('helps', 'VBZ'), ('students', 'NNS'), ('learn', 'NN'), ('.', '.')]
[('I', 'PRP'), ('enjoy', 'VBP'), ('working', 'VBG'), ('with', 'IN'), ('data', 'NNS'), ('models', 'NNS'), ('.', '.')]
[('Artificial', 'JJ'), ('intelligence', 'NN'), ('is', 'VBZ'), ('powerful', 'JJ'), ('.', '.')]


In [ ]:
def generate_random(hmm_model, length=10):
    sentence = []

    # Get all possible states (POS tags) from the model
    all_states = list(hmm_model._states)
    if not all_states:
        print("No states available in HMM model to start generation.")
        return []

    # initial hidden state (POS tag)
    state = random.choice(all_states)

    for _ in range(length):
        # Emission: state -> word
        current_state_outputs = hmm_model._outputs[state]
        output_samples = list(current_state_outputs.samples())
        if output_samples:
            word = random.choice(output_samples)
        else:
            # If no words can be emitted from this state, terminate sentence generation
            print(f"No words can be emitted from state '{state}'. Terminating sentence generation.")
            break
        sentence.append(word)

        # Transition: state -> next state
        current_state_transitions = hmm_model._transitions[state]
        transition_samples = list(current_state_transitions.samples())
        if transition_samples:
            state = random.choice(transition_samples)
        else:
            # If no next state can be transitioned to from this state, terminate sentence generation
            print(f"No transitions from state '{state}'. Terminating sentence generation.")
            break

    return sentence

In [ ]:
generated = generate_random(hmm_model, 10)
print("Generated sentence:")
print(" ".join(generated))

No transitions from state '.'. Terminating sentence generation.
Generated sentence:
Artificial learn science Machine intelligence helps very interesting .


# **WORD PREDICTION USING BROWN CORPUS DATA**

In [ ]:
import nltk
import random
from nltk.corpus import brown
from nltk.tokenize import word_tokenize
from nltk.tag import HiddenMarkovModelTagger

nltk.download('brown')
nltk.download('universal_tagset')
nltk.download('punkt')

print("NLTK resources downloaded successfully.")

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...


NLTK resources downloaded successfully.


[nltk_data]   Package punkt is already up-to-date!


In [ ]:
# Load Brown corpus with universal POS tags
brown_tagged_sents = brown.tagged_sents(tagset='universal')

# Convert to list and shuffle
brown_tagged_sents = list(brown_tagged_sents)
random.shuffle(brown_tagged_sents)

# Train-test split (80-20)
split_point = int(0.8 * len(brown_tagged_sents))
train_sents = brown_tagged_sents[:split_point]
test_sents = brown_tagged_sents[split_point:]

print("Training sentences:", len(train_sents))
print("Testing sentences:", len(test_sents))

Training sentences: 45872
Testing sentences: 11468


In [ ]:
hmm_tagger = HiddenMarkovModelTagger.train(train_sents)

print("HMM trained successfully.")

HMM trained successfully.


In [ ]:
accuracy = hmm_tagger.accuracy(test_sents)
print(f"HMM POS Tagging Accuracy: {accuracy:.4f}")

HMM POS Tagging Accuracy: 0.9510


In [ ]:
# Extract tag set and vocabulary from training data
all_tags = sorted({tag for sent in train_sents for _, tag in sent})
all_words = sorted({word.lower() for sent in train_sents for word, _ in sent})

def predict_next_word(hmm_tagger, last_tag, all_tags, all_words):
    max_prob = 0.0
    best_word = None

    for next_tag in all_tags:
        # Transition probability P(next_tag | last_tag)
        if last_tag in hmm_tagger._transitions.conditions():
            trans_prob = hmm_tagger._transitions[last_tag].prob(next_tag)
        else:
            continue

        if trans_prob == 0:
            continue

        # Emission probability P(word | next_tag)
        if next_tag not in hmm_tagger._outputs.conditions():
            continue

        for word in all_words:
            emit_prob = hmm_tagger._outputs[next_tag].prob(word)
            prob = trans_prob * emit_prob

            if prob > max_prob:
                max_prob = prob
                best_word = word

    return best_word

In [ ]:
input_sentence = "The data science is"

tokens = word_tokenize(input_sentence)
tagged_sentence = hmm_tagger.tag(tokens)

print("\nInput Sentence:", input_sentence)
print("Tagged Sentence:")
print(tagged_sentence)

# Get last POS tag
last_tag = tagged_sentence[-1][1]
print("Last POS tag:", last_tag)


Input Sentence: The data science is
Tagged Sentence:
[('The', 'DET'), ('data', 'NOUN'), ('science', 'NOUN'), ('is', 'VERB')]
Last POS tag: VERB


In [ ]:
predicted_word = predict_next_word(
    hmm_tagger,
    last_tag,
    all_tags,
    all_words
)

print("\nPredicted Next Word:", predicted_word)


Predicted Next Word: the


# **N-GRAM**

In [ ]:
# Load Brown corpus sentences
sentences = brown.sents()

# Convert words to lowercase
sentences = [[word.lower() for word in sent] for sent in sentences]

print("Total sentences:", len(sentences))

Total sentences: 57340


In [ ]:
from collections import defaultdict, Counter

# Bigram counts
bigram_counts = defaultdict(Counter)

# Trigram counts
trigram_counts = defaultdict(Counter)

for sent in sentences:
    for i in range(len(sent) - 1):
        bigram_counts[sent[i]][sent[i + 1]] += 1

    for i in range(len(sent) - 2):
        trigram_counts[(sent[i], sent[i + 1])][sent[i + 2]] += 1

In [ ]:
def predict_next_word_bigram(word, bigram_counts):
    word = word.lower()
    if word not in bigram_counts:
        return None
    return bigram_counts[word].most_common(1)[0][0]

In [ ]:
def predict_next_word_trigram(word1, word2, trigram_counts):
    key = (word1.lower(), word2.lower())
    if key not in trigram_counts:
        return None
    return trigram_counts[key].most_common(1)[0][0]

In [ ]:
input_sentence = "the data science is"
tokens = word_tokenize(input_sentence.lower())

print("Input sentence:", input_sentence)

# Bigram prediction (uses last word)
bigram_prediction = predict_next_word_bigram(tokens[-1], bigram_counts)

# Trigram prediction (uses last two words)
trigram_prediction = predict_next_word_trigram(tokens[-2], tokens[-1], trigram_counts)

print("Bigram predicted next word:", bigram_prediction)
print("Trigram predicted next word:", trigram_prediction)

Input sentence: the data science is
Bigram predicted next word: a
Trigram predicted next word: fully
